# 02 - Data Preparation, Cleaning & Chronological Splitting
## E-Commerce Product Recommendation System (Retailrocket Dataset)

**Objective:**
Transform the raw event logs into reproducible, high-quality analytical datasets ready for machine learning:
1. Validate essential entity IDs and filter invalid records.
2. Standardize Unix millisecond timestamps into UTC datetimes.
3. Investigate duplicate events (accidental double-clicks vs. legitimate repeated views).
4. Assign implicit behavioral confidence weights (`view=1.0`, `addtocart=3.0`, `transaction=5.0`).
5. Aggregate interactions at the `(visitorid, itemid)` level.
6. Execute strict chronological train/val/test splits (70% / 15% / 15%) guaranteeing zero temporal leakage.
7. Export optimized Parquet files into `data/processed/`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.load_data import load_events
from src.data.preprocess import clean_events, split_chronological
from src.features.build_features import aggregate_user_item_interactions, build_popularity_features

DATA_DIR = ROOT_DIR / 'dataset'
OUT_DIR = ROOT_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Ready to process from: {DATA_DIR} to {OUT_DIR}')

Ready to process from: D:\projects & datasets\Ecommerce_rcmd_sys\dataset to D:\projects & datasets\Ecommerce_rcmd_sys\data\processed


### 1. Ingest Raw Events

In [2]:
raw_events = load_events(DATA_DIR)
print(f'Ingested Raw Events: {len(raw_events):,}')
display(raw_events.head())

2026-11-20 02:45:23,751 [INFO] Loading events from D:\projects & datasets\Ecommerce_rcmd_sys\dataset\events.csv (nrows=None)...
2026-11-20 02:45:24,615 [INFO] Loaded 2756101 event records via DuckDB


Ingested Raw Events: 2,756,101


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,<NA>
1,1433224214164,992329,view,248676,<NA>
2,1433221999827,111016,view,318965,<NA>
3,1433221955914,483717,view,253185,<NA>
4,1433221337106,951259,view,367447,<NA>


### 2. Execute Data Cleaning & Standardization
We apply our reproducible cleaning rules via `clean_events()`:
- Check essential non-nullity (`visitorid`, `itemid`, `timestamp`, `event`).
- Deduplicate identical rapid clicks at the exact same millisecond.
- Cast timestamps to UTC datetimes and extract temporal features.

In [3]:
clean_df, cleaning_meta = clean_events(raw_events)
print('\n--- CLEANING SUMMARY ---')
for k, v in cleaning_meta.items():
    print(f'{k}: {v}')

# Save cleaned events
clean_path = OUT_DIR / 'events_clean.parquet'
clean_df.to_parquet(clean_path, index=False)
print(f'Saved cleaned events to: {clean_path}')

2026-11-20 02:45:24,634 [INFO] Starting data cleaning on 2756101 records...
2026-11-20 02:45:26,556 [INFO] Cleaning finished: Retained 2755641 / 2756101 records (99.98%)



--- CLEANING SUMMARY ---
initial_rows: 2756101
final_rows: 2755641
dropped_missing_keys: 0
dropped_invalid_events: 0
dropped_exact_duplicates: 460
retention_rate_pct: 99.9833
min_datetime: 2015-05-03 03:00:04.384000+00:00
max_datetime: 2015-09-18 02:59:47.788000+00:00
unique_visitors: 1407580
unique_items: 235061
Saved cleaned events to: D:\projects & datasets\Ecommerce_rcmd_sys\data\processed\events_clean.parquet


### 3. Aggregate User-Product Interactions & Implicit Feedback Scoring
We calculate behavioral weights:
- View: $w = 1.0$
- Add-to-cart: $w = 3.0$
- Transaction: $w = 5.0$

Total weight per `(visitorid, itemid)` pair represents our implicit interaction confidence.

In [4]:
user_item_df = aggregate_user_item_interactions(clean_df)
print(f'Aggregated User-Item Interactions: {len(user_item_df):,}')
display(user_item_df.head(10))

user_item_path = OUT_DIR / 'user_item_interactions.parquet'
user_item_df.to_parquet(user_item_path, index=False)
print(f'Saved user-item interactions to: {user_item_path}')

2026-11-20 02:45:27,500 [INFO] Aggregating 2755641 event interactions by user-product pairs...
2026-11-20 02:45:29,911 [INFO] Generated 2145179 unique user-product interaction records


Aggregated User-Item Interactions: 2,145,179


,visitorid,itemid,total_weight,event_count,view_count,cart_count,purchase_count,last_timestamp
0,0,67045,1.0,1,1,0,0,1442004917175
1,0,285930,1.0,1,1,0,0,1442004589439
2,0,357564,1.0,1,1,0,0,1442004759591
3,1,72028,1.0,1,1,0,0,1439487966444
4,2,216305,2.0,2,2,0,0,1438971463170
5,2,259884,1.0,1,1,0,0,1438970212664
6,2,325215,3.0,3,3,0,0,1438971657845
7,2,342816,2.0,2,2,0,0,1438971444375
8,3,385090,1.0,1,1,0,0,1438413035296
9,4,177677,1.0,1,1,0,0,1442352267167


Saved user-item interactions to: D:\projects & datasets\Ecommerce_rcmd_sys\data\processed\user_item_interactions.parquet


### 4. Chronological Splitting (70% Train, 15% Validation, 15% Test)
To guarantee zero future information leakage, we enforce a strict temporal boundary split:

In [5]:
train_df, val_df, test_df, split_meta = split_chronological(clean_df, 0.70, 0.15, 0.15)
print('\n--- CHRONOLOGICAL SPLIT SUMMARY ---')
for k, v in split_meta.items():
    print(f'{k}: {v}')

# Verify temporal boundaries
assert train_df['timestamp'].max() < val_df['timestamp'].min(), 'Leakage detected!'
assert val_df['timestamp'].max() < test_df['timestamp'].min(), 'Leakage detected!'
print('\n[PASS] Zero data leakage assertion verified successfully.')

# Export splits to Parquet
train_df.to_parquet(OUT_DIR / 'train.parquet', index=False)
val_df.to_parquet(OUT_DIR / 'val.parquet', index=False)
test_df.to_parquet(OUT_DIR / 'test.parquet', index=False)
print(f'Splits successfully written to {OUT_DIR}')

2026-11-20 02:45:30,942 [INFO] Chronological split completed cleanly with zero temporal leakage.



--- CHRONOLOGICAL SPLIT SUMMARY ---
train_rows: 1928948
val_rows: 413346
test_rows: 413347
train_pct: 70.0
val_pct: 15.0
test_pct: 15.0
train_end_time: 2015-08-02 23:29:02.354000+00:00
val_start_time: 2015-08-02 23:29:03.291000+00:00
val_end_time: 2015-08-25 22:30:59.619000+00:00
test_start_time: 2015-08-25 22:31:01.629000+00:00
test_end_time: 2015-09-18 02:59:47.788000+00:00
train_unique_users: 978906
train_unique_items: 200974
val_unique_users: 234715
test_unique_users: 233728

[PASS] Zero data leakage assertion verified successfully.
Splits successfully written to D:\projects & datasets\Ecommerce_rcmd_sys\data\processed


### 5. Popularity Features (Strictly on Train Split)

In [6]:
pop_df = build_popularity_features(train_df)
print('Top 10 Most Popular Products in Training Period:')
display(pop_df.head(10))
pop_df.to_parquet(OUT_DIR / 'popularity_features.parquet', index=False)
print(f'Saved popularity features to: {OUT_DIR / "popularity_features.parquet"}')

2026-11-20 02:45:31,870 [INFO] Computing global popularity features on 1928948 training records...
2026-11-20 02:45:32,636 [INFO] Computed popularity metrics for 200974 distinct training items


Top 10 Most Popular Products in Training Period:


,itemid,popularity_score,unique_visitors,total_views,total_carts,total_purchases,popularity_rank
0,461686,2142.0,764,1317,165,66,1
1,5411,1903.0,1688,1879,8,0,2
2,309778,1754.0,1037,1535,48,15,3
3,370653,1483.0,1254,1483,0,0,4
4,257040,1477.0,802,1218,58,17,5
5,369447,1475.0,595,1080,80,31,6
6,7943,1443.0,619,1029,78,36,7
7,298009,1347.0,1091,1347,0,0,8
8,48030,1268.0,516,826,84,38,9
9,335975,1233.0,1004,1233,0,0,10


Saved popularity features to: D:\projects & datasets\Ecommerce_rcmd_sys\data\processed\popularity_features.parquet
